# Deep dive 1/13: `customers`

Checks whether `silver.dim_customers` correctly implements every transformation `bronze.customers` actually needs -- not just "does it run," but "is each column typed, cleaned, and validated the way the data actually requires."

Source of truth for what Silver *should* do: `data_pipelines/silver/table_specs.py`'s `customers_spec` and `data_pipelines/silver/transforms.py`. Source of truth for what the data *actually* looks like: this notebook, run against your real `data/latam_bank.duckdb`.

Run top to bottom. Section 10 is a checklist to fill in with what you found; free cells are at the end for anything this doesn't already cover.

In [1]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 1000)
pd.set_option("display.min_rows", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 100)

con = duckdb.connect("data/latam_bank.duckdb")

def q(sql, params=None):
    """Run a query and return a pandas DataFrame."""
    return con.execute(sql, params or []).df()

def q_full(sql, params=None):
    """Like q(), but shows every cell's full text with no truncation."""
    with pd.option_context("display.max_colwidth", None):
        return q(sql, params)

## 1. Column inventory -- does `silver.dim_customers` have everything `customers_spec` says it should?

This isn't "does the table exist" (you already know it does from `run_silver.py`'s output) -- it's "does every column `table_specs.py` declares actually show up with the right name," which catches a spec/build drift that a successful run wouldn't otherwise surface.

In [2]:
# Exact column list customers_spec declares (target names, i.e. what should land in Silver),
# transcribed from data_pipelines/silver/table_specs.py's customers_spec -- keep this in sync if
# that spec changes.
expected_silver_columns = [
    "customer_id", "document_number", "document_type", "first_name", "last_name",
    "date_of_birth", "gender", "email", "mobile_phone", "landline_phone", "address",
    "city", "state", "country", "postal_code", "detected_accent", "segment",
    "credit_score", "estimated_monthly_income", "occupation", "marital_status",
    "education_level", "registration_date", "registration_branch_id", "customer_status",
    "last_updated", "accepts_marketing",
]

actual_silver_columns = q("""
    SELECT column_name FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='dim_customers'
    ORDER BY ordinal_position
""")["column_name"].tolist()

missing = [c for c in expected_silver_columns if c not in actual_silver_columns]
extra = [c for c in actual_silver_columns if c not in expected_silver_columns]
print("Missing from silver.dim_customers (spec declares, table lacks):", missing or "none")
print("Extra in silver.dim_customers (table has, spec doesn't declare):", extra or "none")
print()
print("Column order as built:", actual_silver_columns)

Missing from silver.dim_customers (spec declares, table lacks): none
Extra in silver.dim_customers (table has, spec doesn't declare): none

Column order as built: ['customer_id', 'document_number', 'document_type', 'first_name', 'last_name', 'date_of_birth', 'gender', 'email', 'mobile_phone', 'landline_phone', 'address', 'city', 'state', 'country', 'postal_code', 'detected_accent', 'segment', 'credit_score', 'estimated_monthly_income', 'occupation', 'marital_status', 'education_level', 'registration_date', 'registration_branch_id', 'customer_status', 'last_updated', 'accepts_marketing']


In [3]:
# Bronze's raw columns, for reference -- everything below this line should be accounted for by
# either a Silver column or a deliberate drop (Hive partition columns, _ingested_at, etc.)
bronze_cols = q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='bronze' AND table_name='customers'
    ORDER BY ordinal_position
""")
bronze_cols

,column_name,data_type
0,customer_id,VARCHAR
1,document_number,VARCHAR
2,document_type,VARCHAR
3,first_name,VARCHAR
4,last_name,VARCHAR
5,date_of_birth,VARCHAR
6,gender,VARCHAR
7,email,VARCHAR
8,mobile_phone,VARCHAR
9,landline_phone,VARCHAR


In [4]:
# Silver columns
bronze_cols = q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='dim_customers'
    ORDER BY ordinal_position
""")
bronze_cols

,column_name,data_type
0,customer_id,VARCHAR
1,document_number,VARCHAR
2,document_type,VARCHAR
3,first_name,VARCHAR
4,last_name,VARCHAR
5,date_of_birth,DATE
6,gender,VARCHAR
7,email,VARCHAR
8,mobile_phone,VARCHAR
9,landline_phone,VARCHAR


## 2. Row-count reconciliation -- did dedup do what it's supposed to?

`silver.dim_customers` should have exactly one row per distinct `customer_id`, keeping the latest `_ingested_at`. This is the same invariant the quality gate's `silver_row_delta_unexplained` check enforces, reproduced here directly against your data.

In [5]:
bronze_total = q("SELECT count(*) AS n FROM bronze.customers")["n"][0]
bronze_distinct_pk = q("SELECT count(DISTINCT customer_id) AS n FROM bronze.customers")["n"][0]
silver_total = q("SELECT count(*) AS n FROM silver.dim_customers")["n"][0]
silver_distinct_pk = q("SELECT count(DISTINCT customer_id) AS n FROM silver.dim_customers")["n"][0]

print(f"bronze.customers:      {bronze_total:,} rows, {bronze_distinct_pk:,} distinct customer_id")
print(f"silver.dim_customers:  {silver_total:,} rows, {silver_distinct_pk:,} distinct customer_id")
print()
print("Duplicate raw rows in Bronze (resends/reloads):", bronze_total - bronze_distinct_pk)
print("OK -- Silver row count matches Bronze distinct PK count" if silver_total == bronze_distinct_pk
      else "MISMATCH -- dedup did not produce one row per customer_id, investigate")

bronze.customers:      150,000 rows, 150,000 distinct customer_id
silver.dim_customers:  150,000 rows, 150,000 distinct customer_id

Duplicate raw rows in Bronze (resends/reloads): 0
OK -- Silver row count matches Bronze distinct PK count


In [6]:
# Any duplicate customer_id left in Silver itself? Should always be empty.
q("""
    SELECT customer_id, count(*) AS n
    FROM silver.dim_customers
    GROUP BY 1 HAVING count(*) > 1
""")

,customer_id,n


## 3. NOT NULL audit -- does Bronze actually satisfy the data dictionary's constraints?

The data dictionary marks these columns `NOT NULL`: `customer_id`, `document_number`, `document_type`, `first_name`, `last_name`, `date_of_birth`, `city`, `state`, `country`, `segment`, `registration_date`, `registration_branch_id`, `customer_status`, `last_updated`, `accepts_marketing`. Silver doesn't enforce this (DuckDB tables here carry no constraints), so if any of these are actually nullable in the real data, Silver will silently carry the NULLs through -- worth knowing before a report assumes these are always populated.

In [7]:
declared_not_null = [
    "customer_id", "document_number", "document_type", "first_name", "last_name",
    "date_of_birth", "city", "state", "country", "segment", "registration_date",
    "registration_branch_id", "customer_status", "last_updated", "accepts_marketing",
]

rows = []
total = q("SELECT count(*) AS n FROM bronze.customers")["n"][0]
for col in declared_not_null:
    nulls = q(f"""
        SELECT count(*) AS n FROM bronze.customers
        WHERE {col} IS NULL OR trim(CAST({col} AS VARCHAR)) = ''
    """)["n"][0]
    rows.append({"column": col, "declared": "NOT NULL", "actual_nulls": nulls,
                 "null_rate": round(nulls / total, 4) if total else None})

null_audit = pd.DataFrame(rows)
null_audit

,column,declared,actual_nulls,null_rate
0,customer_id,NOT NULL,0,0.0
1,document_number,NOT NULL,0,0.0
2,document_type,NOT NULL,0,0.0
3,first_name,NOT NULL,0,0.0
4,last_name,NOT NULL,0,0.0
5,date_of_birth,NOT NULL,0,0.0
6,city,NOT NULL,0,0.0
7,state,NOT NULL,0,0.0
8,country,NOT NULL,0,0.0
9,segment,NOT NULL,0,0.0


## 4. Domain / categorical checks

Compares actual distinct values against what the data dictionary declares, for every categorical-ish column. A value outside the declared set isn't necessarily wrong (the dictionary itself admits synthetic imperfections), but Silver doesn't flag or filter any of these except `customer_status` (already checked by the quality gate) -- worth seeing what the others actually contain.

In [8]:
# document_type -- dictionary: DNI, CURP, CC, CE, Passport
q("SELECT document_type, count(*) AS n FROM bronze.customers GROUP BY 1 ORDER BY n DESC")

,document_type,n
0,DNI,104749
1,CE,15150
2,Pasaporte,15062
3,CC,15039


In [9]:
# gender -- dictionary: M, F, O
q("SELECT gender, count(*) AS n FROM bronze.customers GROUP BY 1 ORDER BY n DESC")

,gender,n
0,F,50508
1,O,49808
2,M,49684


In [10]:
# segment -- dictionary: Premium, Plus, Basic, Student
q("SELECT segment, count(*) AS n FROM bronze.customers GROUP BY 1 ORDER BY n DESC")

,segment,n
0,Basic,89756
1,Plus,37547
2,Premium,15207
3,Student,7490


In [11]:
# customer_status -- dictionary: Active, Inactive, Suspended, Closed
# NOTE: table_specs.py/quality's domain check actually uses {Active, Inactive, Blocked, Closed} --
# "Suspended" (dictionary) vs "Blocked" (contract) is worth reconciling; this cell shows the raw
# truth either way. This is the known warning from the quality report (4,407/150,000).
q("SELECT customer_status, count(*) AS n FROM bronze.customers GROUP BY 1 ORDER BY n DESC")

,customer_status,n
0,Active,127700
1,Inactive,14914
2,Suspended,4407
3,Closed,2979


In [12]:
# marital_status / education_level -- not declared in the dictionary's domain list at all,
# so there's no "correct" set to check against -- just seeing what's actually there.
print("marital_status:")
display(q("SELECT marital_status, count(*) AS n FROM bronze.customers GROUP BY 1 ORDER BY n DESC"))
print("education_level:")
display(q("SELECT education_level, count(*) AS n FROM bronze.customers GROUP BY 1 ORDER BY n DESC"))

marital_status:


,marital_status,n
0,Married,34765
1,Divorced,34734
2,Single,34474
3,Widowed,34072
4,NaN,11955


education_level:


,education_level,n
0,College Prep,39583
1,High School,33120
2,University,32870
3,NaN,17952
4,Graduate,13269
5,Elementary,13206


In [13]:
# marital_status / education_level -- not declared in the dictionary's domain list at all,
# so there's no "correct" set to check against -- just seeing what's actually there.
print("marital_status:")
display(q("SELECT marital_status, count(*) AS n FROM silver.dim_customers GROUP BY 1 ORDER BY n DESC"))
print("education_level:")
display(q("SELECT education_level, count(*) AS n FROM silver.dim_customers GROUP BY 1 ORDER BY n DESC"))

marital_status:


,marital_status,n
0,Married,34765
1,Divorced,34734
2,Single,34474
3,Widowed,34072
4,NaN,11955


education_level:


,education_level,n
0,College Prep,39583
1,High School,33120
2,University,32870
3,NaN,17952
4,Graduate,13269
5,Elementary,13206


Users note: are these 'NaN' values in marital_status and education_level acctually 'NaN' string values or are they null?

## 5. Numeric range sanity

`credit_score` and `estimated_monthly_income` are cast to DOUBLE in Silver but never range-checked. The dictionary states `credit_score` should be 300-850 -- worth confirming the real data actually stays inside that band, and that income has no negative/absurd outliers Silver would otherwise pass through untouched.

In [14]:
q("""
    SELECT
        min(TRY_CAST(credit_score AS DOUBLE)) AS min_credit_score,
        max(TRY_CAST(credit_score AS DOUBLE)) AS max_credit_score,
        count(*) FILTER (WHERE TRY_CAST(credit_score AS DOUBLE) NOT BETWEEN 300 AND 850) AS out_of_declared_range,
        count(*) FILTER (WHERE credit_score IS NULL) AS original_nulls,  -- users note
        count(*) FILTER (WHERE trim(credit_score) = '') AS empty_strings_nulls,  -- users note
        count(*) FILTER (WHERE credit_score IS NOT NULL and TRY_CAST(credit_score AS DOUBLE) IS NULL) AS casting_fail_nulls  -- users note
    FROM bronze.customers
""")

,min_credit_score,max_credit_score,out_of_declared_range,original_nulls,empty_strings_nulls,casting_fail_nulls
0,422.0,850.0,0,22492,0,0


In [15]:
q("""
    SELECT
        typeof(credit_score), -- users note: check if data type is casted in silver
        min(credit_score) AS min_credit_score,
        max(credit_score) AS max_credit_score,
        count(*) FILTER (WHERE credit_score NOT BETWEEN 300 AND 850) AS out_of_declared_range,
        count(*) FILTER (WHERE credit_score IS NULL) AS nulls
    FROM silver.dim_customers
    GROUP BY ALL
""")

,typeof(credit_score),min_credit_score,max_credit_score,out_of_declared_range,nulls
0,DOUBLE,422.0,850.0,0,22492


In [16]:
q("""
    SELECT
        min(TRY_CAST(estimated_monthly_income AS DOUBLE)) AS min_income,
        max(TRY_CAST(estimated_monthly_income AS DOUBLE)) AS max_income,
        count(*) FILTER (WHERE TRY_CAST(estimated_monthly_income AS DOUBLE) < 0) AS negative_income_rows,
        count(*) FILTER (WHERE estimated_monthly_income IS NULL) AS original_nulls,  -- users note
        count(*) FILTER (WHERE trim(estimated_monthly_income) = '') AS empty_strings_nulls,  -- users note
        count(*) FILTER (WHERE estimated_monthly_income IS NOT NULL and TRY_CAST(estimated_monthly_income AS DOUBLE) IS NULL) AS casting_fail_nulls  -- users note
    FROM bronze.customers
""")

,min_income,max_income,negative_income_rows,original_nulls,empty_strings_nulls,casting_fail_nulls
0,5100.28,1.118951e+08,0,30033,0,0


In [17]:
q("""
    SELECT
        typeof(estimated_monthly_income), -- users note: check if data type is casted in silver
        min(TRY_CAST(estimated_monthly_income AS DOUBLE)) AS min_income,
        max(TRY_CAST(estimated_monthly_income AS DOUBLE)) AS max_income,
        count(*) FILTER (WHERE TRY_CAST(estimated_monthly_income AS DOUBLE) < 0) AS negative_income_rows,
        count(*) FILTER (WHERE estimated_monthly_income IS NULL) AS nulls
    FROM silver.dim_customers
    GROUP BY ALL
""")

,typeof(estimated_monthly_income),min_income,max_income,negative_income_rows,nulls
0,DOUBLE,5100.28,1.118951e+08,0,30033


## 6. Date sanity

Three checks Silver's plain `TRY_CAST(... AS DATE/TIMESTAMP)` doesn't do: implausible ages from `date_of_birth`, `registration_date` falling outside the dataset's stated window (June 2023 - June 2026 per the data dictionary), and the `last_updated` anomaly the spec's own docstring already flags (a 2027 value against a dataset that otherwise runs through 2026-06-17).

In [18]:
# Age sanity from date_of_birth -- flags anyone implausibly old (>100) or with a birth date in
# the future/inside the dataset window (implying a data entry error, not a real customer).
q("""
    SELECT
        min(TRY_CAST(date_of_birth AS DATE)) AS earliest_dob,
        max(TRY_CAST(date_of_birth AS DATE)) AS latest_dob,
        count(*) FILTER (WHERE date_diff('year', TRY_CAST(date_of_birth AS DATE), current_date) > 100) AS implausibly_old,
        count(*) FILTER (WHERE TRY_CAST(date_of_birth AS DATE) > current_date) AS future_birth_dates,
        count(*) FILTER (WHERE date_of_birth IS NULL OR trim(date_of_birth) = '' or TRY_CAST(date_of_birth AS DATE) is null) AS nulls
    FROM bronze.customers
""")

,earliest_dob,latest_dob,implausibly_old,future_birth_dates,nulls
0,1942-07-07,2005-06-21,0,0,0


In [19]:
# registration_date vs the dataset's stated window (2023-06-17 to 2026-06-17)
q("""
    SELECT
        min(TRY_CAST(registration_date AS TIMESTAMP)) AS earliest_registration,
        max(TRY_CAST(registration_date AS TIMESTAMP)) AS latest_registration,
        count(*) FILTER (WHERE TRY_CAST(registration_date AS TIMESTAMP) < TIMESTAMP '2023-06-17') AS before_window,
        count(*) FILTER (WHERE TRY_CAST(registration_date AS TIMESTAMP) > TIMESTAMP '2026-06-17') AS after_window
    FROM bronze.customers
""")

,earliest_registration,latest_registration,before_window,after_window
0,2018-06-18 01:21:11,2026-06-17 23:53:29,93695,48


In [20]:
# Users note: check if casting type fails in silver
q("""
    SELECT
        typeof(registration_date),
        count(*) FILTER (WHERE registration_date is null)
    FROM silver.dim_customers
    GROUP BY ALL
""")

,typeof(registration_date),count_star() FILTER (WHERE (registration_date IS NULL))
0,TIMESTAMP,0


In [21]:
# last_updated -- the spec docstring already flags a 2027 value; see exactly how many and how far out.
q("""
    SELECT
        max(TRY_CAST(last_updated AS TIMESTAMP)) AS latest_last_updated,
        count(*) FILTER (WHERE TRY_CAST(last_updated AS TIMESTAMP) > TIMESTAMP '2026-06-17') AS after_dataset_window,
        count(*) FILTER (WHERE TRY_CAST(last_updated AS TIMESTAMP) < TRY_CAST(registration_date AS TIMESTAMP)) AS updated_before_registered
    FROM bronze.customers
""")

,latest_last_updated,after_dataset_window,updated_before_registered
0,2027-06-15 19:35:27,9379,0


In [22]:
# What do the after-window last_updated rows actually look like?
q("""
    SELECT customer_id, registration_date, last_updated
    FROM bronze.customers
    WHERE TRY_CAST(last_updated AS TIMESTAMP) > TIMESTAMP '2026-06-17'
    ORDER BY last_updated DESC
    LIMIT 20
""")

,customer_id,registration_date,last_updated
0,CLI-L46PYG6JJZND,2026-06-16 19:35:27,2027-06-15 19:35:27
1,CLI-VLZGTZ7OZSTT,2026-06-16 08:10:12,2027-06-10 08:10:12
2,CLI-HJPMWDLZ9DVG,2026-06-16 00:03:01,2027-06-10 00:03:01
3,CLI-LN892ED21IAO,2026-06-13 13:18:43,2027-06-09 13:18:43
4,CLI-OECKCKFE6WYJ,2026-06-12 06:55:59,2027-06-07 06:55:59
5,CLI-Q0T8KUMAVITH,2026-06-12 19:01:40,2027-06-06 19:01:40
6,CLI-HVYT8OIBDUPC,2026-06-11 00:10:17,2027-06-06 00:10:17
7,CLI-TNJQ8VPVYKCT,2026-06-08 21:00:36,2027-06-05 21:00:36
8,CLI-AM5K97668UNZ,2026-06-06 16:38:54,2027-06-05 16:38:54
9,CLI-7DJHGJMIQ2K9,2026-06-10 16:09:20,2027-06-05 16:09:20


## 7. Country canonicalization -- did `as_country()` actually resolve every spelling variant?

`COUNTRY_CANONICAL` in `config.py` maps known spelling splits (`mexico`/`méxico` -> `México`, etc.) to one canonical value. This checks whether *every* distinct raw value in `bronze.customers.country` is covered by that map -- an uncovered value falls through unchanged per `as_country()`'s own design, so it wouldn't error, it would just silently stay split in `silver.dim_customers.country`.

In [23]:
print("Distinct raw country values in Bronze:")
display(q("SELECT country, count(*) AS n FROM bronze.customers GROUP BY 1 ORDER BY n DESC"))

print("Distinct country values in Silver (post-canonicalization):")
display(q("SELECT country, count(*) AS n FROM silver.dim_customers GROUP BY 1 ORDER BY n DESC"))

Distinct raw country values in Bronze:


,country,n
0,México,74907
1,Colombia,45251
2,Argentina,29842


Distinct country values in Silver (post-canonicalization):


,country,n
0,México,74907
1,Colombia,45251
2,Argentina,29842


In [24]:
# The canonicalization map itself, mirrored from config.py -- keep in sync if that dict changes.
country_canonical = {
    "méxico": "México", "mexico": "México", "colombia": "Colombia", "argentina": "Argentina",
    "usa": "USA", "united states": "USA", "spain": "Spain", "brazil": "Brazil", "brasil": "Brazil",
}

raw_values = q("SELECT DISTINCT lower(trim(country)) AS v FROM bronze.customers WHERE country IS NOT NULL")["v"].tolist()
uncovered = [v for v in raw_values if v not in country_canonical]
print("Raw country values NOT covered by COUNTRY_CANONICAL (pass through unchanged):", uncovered or "none -- full coverage")

Raw country values NOT covered by COUNTRY_CANONICAL (pass through unchanged): none -- full coverage


In [25]:
# Users note: list of cities
q("SELECT len(city) as len, city, count(*) AS n FROM bronze.customers GROUP BY all ORDER BY len")

,len,city,n
0,4,Cali,8972
1,6,Puebla,12400
2,6,Bogotá,9140
3,7,Córdoba,5792
4,7,Tijuana,12489
5,7,Mendoza,5959
6,7,Rosario,6081
7,8,La Plata,5973
8,8,Medellín,9107
9,9,Monterrey,12369


## 8. `postal_code` -- confirm the "kept as VARCHAR" reasoning

`table_specs.py`'s docstring says `postal_code` stays VARCHAR because only ~72.06% of values are numeric-parseable. Confirming that number directly against real data, since a stale profile claim (from synthetic dev data, not your production pull) could be wrong.

In [26]:
q("""
    SELECT
        count(*) AS total,
        count(*) FILTER (WHERE TRY_CAST(postal_code AS BIGINT) IS NOT NULL) AS numeric_parseable,
        round(100.0 * count(*) FILTER (WHERE TRY_CAST(postal_code AS BIGINT) IS NOT NULL) / count(*), 2) AS pct_numeric
    FROM bronze.customers
    WHERE postal_code IS NOT NULL AND trim(postal_code) <> ''
""")

,total,numeric_parseable,pct_numeric
0,134956,108088,80.09


## 9. `accepts_marketing` -- did boolean parsing behave as expected?

`as_boolean()` only recognizes literal `'true'`/`'false'` (case-insensitive); anything else becomes NULL rather than a guessed default. Comparing Bronze's raw distinct values against Silver's parsed boolean distribution shows exactly what got parsed vs. what became NULL and why.

In [27]:
print("Bronze raw values:")
display(q("SELECT typeof(accepts_marketing), accepts_marketing, count(*) AS n FROM bronze.customers GROUP BY ALL ORDER BY n DESC"))

print("Silver parsed values:")
display(q("SELECT typeof(accepts_marketing), accepts_marketing, count(*) AS n FROM silver.dim_customers GROUP BY ALL ORDER BY n DESC"))

# Users note: added the typeof to check if the typecasting is correct 

Bronze raw values:


,typeof(accepts_marketing),accepts_marketing,n
0,VARCHAR,False,75007
1,VARCHAR,True,74993


Silver parsed values:


,typeof(accepts_marketing),accepts_marketing,n
0,BOOLEAN,False,75007
1,BOOLEAN,True,74993


## 10.5 Follow-up checks -- closing the gap with the `products` deep dive

Added after review: `document_number`'s UNIQUE constraint was never checked (only NOT NULL), and `email`/`mobile_phone`/`landline_phone`/`address`/`detected_accent`/`occupation` had no checks at all. `state` only had a NOT NULL check, no cross-check against `country`. These four cells close that gap.

In [28]:
# document_number -- dictionary: NOT NULL, UNIQUE (same constraint products_spec's product_number
# has, which the products notebook checks directly -- this was missing here).
q("""
    SELECT
        count(*) AS total,
        count(DISTINCT document_number) AS distinct_document_number,
        count(*) FILTER (WHERE document_number IS NULL OR trim(document_number) = '') AS nulls
    FROM bronze.customers
""")

,total,distinct_document_number,nulls
0,150000,150000,0


In [29]:
# If any document_number repeats, is it the same customer_id each time (a resend/dup row, harmless)
# or two different customers sharing a document number (a real uniqueness violation)?
q("""
    SELECT document_number, count(*) AS n, count(DISTINCT customer_id) AS distinct_customer_ids
    FROM bronze.customers
    GROUP BY 1 HAVING count(*) > 1
    ORDER BY n DESC
    LIMIT 20
""")

,document_number,n,distinct_customer_ids


In [30]:
# email / mobile_phone / landline_phone / address -- no dictionary-declared format, so this is
# basic shape sanity rather than a pass/fail check: null rates, and how many populated values look
# structurally wrong (no '@' in email, non-digit-heavy phone numbers).
q("""
    SELECT
        count(*) AS total,
        count(*) FILTER (WHERE email IS NULL OR trim(email) = '') AS email_nulls,
        count(*) FILTER (WHERE email IS NOT NULL AND trim(email) <> '' AND email NOT LIKE '%@%') AS email_missing_at,
        count(*) FILTER (WHERE mobile_phone IS NULL OR trim(mobile_phone) = '') AS mobile_nulls,
        count(*) FILTER (WHERE mobile_phone IS NOT NULL AND trim(mobile_phone) <> ''
                          AND regexp_matches(mobile_phone, '[0-9]')  = false) AS mobile_no_digits,
        count(*) FILTER (WHERE landline_phone IS NULL OR trim(landline_phone) = '') AS landline_nulls,
        count(*) FILTER (WHERE address IS NULL OR trim(address) = '') AS address_nulls
    FROM bronze.customers
""")

,total,email_nulls,email_missing_at,mobile_nulls,mobile_no_digits,landline_nulls,address_nulls
0,150000,2984,0,4707,0,75053,7370


In [31]:
# detected_accent / occupation -- no declared domain either; just seeing the actual value spread
# and null rates, same treatment as marital_status/education_level got originally.
print("detected_accent:")
display(q("SELECT detected_accent, count(*) AS n FROM bronze.customers GROUP BY 1 ORDER BY n DESC"))

print("occupation (top 20 by frequency -- free text, so this is a spot-check, not a full domain):")
display(q("SELECT occupation, count(*) AS n FROM bronze.customers GROUP BY 1 ORDER BY n DESC LIMIT 20"))

print("occupation null rate:")
display(q("SELECT count(*) AS total, count(*) FILTER (WHERE occupation IS NULL OR trim(occupation) = '') AS nulls FROM bronze.customers"))

detected_accent:


,detected_accent,n
0,mexican,52505
1,NaN,44817
2,colombian,31666
3,argentine,21012


occupation (top 20 by frequency -- free text, so this is a spot-check, not a full domain):


,occupation,n
0,NaN,15039
1,Manager,6862
2,Accountant,6857
3,Salesperson,6823
4,Homemaker,6806
5,Entrepreneur,6805
6,Doctor,6794
7,Lawyer,6779
8,Technician,6778
9,Artist,6762


occupation null rate:


,total,nulls
0,150000,15039


In [32]:
# state vs. country -- no declared FK/lookup between them, so this can't be a strict pass/fail,
# but a state value that never co-occurs with a plausible country would be worth a closer look.
# Shows the state/country pairs and their row counts so you can eyeball anything that looks wrong.
q("""
    SELECT country, state, count(*) AS n
    FROM bronze.customers
    GROUP BY 1, 2
    ORDER BY country, n DESC
""")

,country,state,n
0,Argentina,Santa Fe,6081
1,Argentina,Ciudad Autónoma de Buenos Aires,6037
2,Argentina,Buenos Aires,5973
3,Argentina,Mendoza,5959
4,Argentina,Córdoba,5792
5,Colombia,Cundinamarca,9140
6,Colombia,Antioquia,9107
7,Colombia,Atlántico,9076
8,Colombia,Valle del Cauca,8972
9,Colombia,Bolívar,8956


## 10. Known issues already confirmed elsewhere (recap, not re-derived here)

- **`registration_branch_id` -> `branches.branch_id`**: 99.997% orphaned (149,995/150,000). Confirmed a real source-data defect, not a Silver bug or format mismatch -- see `diagnose_branch_fk.ipynb` and the note added to `data_pipelines/silver/README.md`.
- **`customer_status` domain violation**: 4,407/150,000 rows (~2.94%) outside `{Active, Inactive, Blocked, Closed}` -- see section 4 above for exactly which values those are.

## Checklist -- fill in after running the sections above

- [ ] Column inventory (section 1): spec and table agree, no drift
- [ ] Row reconciliation (section 2): Silver row count == Bronze distinct customer_id count
- [ ] NOT NULL audit (section 3): note any column the dictionary claims is NOT NULL but isn't in practice
- [ ] Domain checks (section 4): note any category value outside the declared set, beyond the known `customer_status` warning
- [ ] Numeric ranges (section 5): note any `credit_score` outside 300-850 or negative income
- [ ] Date sanity (section 6): note implausible ages, out-of-window registrations, or `last_updated` anomalies beyond the one already documented
- [ ] Country canonicalization (section 7): confirm `uncovered` list is empty
- [ ] `postal_code` (section 8): confirm the ~72% numeric-parseable claim still holds
- [ ] `accepts_marketing` (section 9): confirm the NULL rate after parsing is explained by genuinely unparseable source values, not a transform bug

## Your own exploration

Free cells below -- `con`, `q()`, and `q_full()` are already set up.

### Users note: Duplicated first and last names, how can we treat them?

In [33]:
q("SELECT len(first_name) as len, min(first_name), max(first_name), count(*) AS n FROM bronze.customers GROUP BY 1 ORDER BY len")

,len,min(first_name),max(first_name),n
0,3,Ana,Luz,2135
1,4,José,Rosa,5390
2,5,César,Óscar,27480
3,6,Alicia,Víctor,28233
4,7,Adriana,Rodrigo,18801
5,8,Ana Rosa,Victoria,15865
6,9,Alejandra,Óscar Luz,8003
7,10,Alicia Ana,Óscar Rosa,1590
8,11,Adriana Ana,Óscar Óscar,4451
9,12,Adriana Rosa,Óscar Víctor,7420


In [34]:
q("SELECT * FROM bronze.customers WHERE first_name in ('Óscar Óscar', 'Adriana Adriana', 'Victoria Victoria', 'Valentina Valentina', 'Alejandra Alejandra')")

,customer_id,document_number,document_type,first_name,last_name,date_of_birth,gender,email,mobile_phone,landline_phone,address,city,state,country,postal_code,detected_accent,segment,credit_score,estimated_monthly_income,occupation,marital_status,education_level,registration_date,registration_branch_id,customer_status,last_updated,accepts_marketing,_source_file,_ingested_at,_source_table
0,CLI-71KEXVRLBR7I,82347609,DNI,Alejandra Alejandra,Miranda Navarro,1970-05-05,F,alejandra.miranda954@hotmail.com,+54 9 23 7114 9879,NaN,"Camino Calle Alsina 194, Centro",La Plata,Buenos Aires,Argentina,B1900,argentine,Plus,790.0,NaN,NaN,Divorced,High School,2023-01-11 04:16:15,SUC-CSY0EVAQ,Inactive,2023-10-26 04:16:15,False,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/customers.csv,2026-09-28 19:22:19.503896-03:00,customers
1,CLI-V7V25WJ8UCSW,32754746,DNI,Óscar Óscar,Cortés Reyes,1944-02-21,M,oscar.cortes@yahoo.com,+54 9 16 3292 5511,+54 48 5050 2535,"Calzada San Juan 467, Centro",Guadalajara,Jalisco,México,NaN,mexican,Plus,705.0,129133.92,Independent Professional,Widowed,University,2025-06-13 21:03:26,SUC-4FVRKONZ,Inactive,2025-08-20 21:03:26,False,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/customers.csv,2026-09-28 19:22:19.503896-03:00,customers
2,CLI-SHGHSJI2GFJE,9868723293,CC,Victoria Victoria,Molina Guerrero,1969-02-17,F,victoriamolina@gmail.com,+57 322 203 1625,NaN,"Diagonal Calle 134 #434, Barrio Sabaneta",Medellín,Antioquia,Colombia,50730,colombian,Basic,627.0,7956862.53,Driver,Widowed,College Prep,2019-09-16 18:17:15,SUC-GRR8NZGN,Active,2020-06-05 18:17:15,False,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/customers.csv,2026-09-28 19:22:19.503896-03:00,customers
3,CLI-XKLH3WQQMRLT,29828768,DNI,Alejandra Alejandra,Delgado Martínez,1963-03-12,O,alejandra.delgado@icloud.com,+54 9 64 5976 7694,NaN,"Calle Guerrero 58, Centro",Puebla,Puebla,México,72000,mexican,Plus,677.0,139812.3,Artist,Married,University,2020-05-09 09:01:23,SUC-IVT82PXM,Active,2021-02-10 09:01:23,False,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/customers.csv,2026-09-28 19:22:19.503896-03:00,customers
4,CLI-Y8R3D12PUTR1,59966902,DNI,Valentina Valentina,Gómez Ortiz,1990-10-24,O,valentina.gomez@outlook.com,+54 9 81 2108 8188,+54 45 7656 4512,"Boulevard Calle Lavalle 52, San Telmo",Buenos Aires,Ciudad Autónoma de Buenos Aires,Argentina,C1000,argentine,Basic,719.0,820713.71,NaN,Divorced,NaN,2021-03-21 06:53:32,SUC-6M96XC6V,Active,2022-01-05 06:53:32,True,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/customers.csv,2026-09-28 19:22:19.503896-03:00,customers
5,CLI-9FSBIIC8TEB7,16126781,DNI,Victoria Victoria,Pérez Aguilar,1969-06-30,F,victoria.perez106@live.com,+54 9 60 2786 1439,NaN,"Calle Del Bosque 288, Centro",Tijuana,Baja California,México,22000,NaN,Basic,623.0,45204.87,Administrative,Divorced,University,2020-11-07 03:28:47,SUC-ZJYKMP1N,Inactive,2020-11-16 03:28:47,True,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/customers.csv,2026-09-28 19:22:19.503896-03:00,customers
6,CLI-VG1VA677DVHR,40132573,DNI,Victoria Victoria,Guerrero Campos,1991-02-17,F,vguerrero@hotmail.com,+54 9 56 4376 7296,NaN,"Avenida Santa María 246, Centro",Querétaro,Querétaro,México,76000,NaN,Premium,823.0,417444.54,Homemaker,Divorced,High School,2019-01-20 19:05:43,SUC-I3AQ0C5S,Active,2019-10-17 19:05:43,False,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/customers.csv,2026-09-28 19:22:19.503896-03:00,customers
7,CLI-AMU293IJGM7R,24148280,DNI,Adriana Adriana,Giménez Castro,1965-09-20,O,agimenez@live.com,+54 9 50 2033 9067,+54 78 2573 8029,"Diagonal Calle San Martín 114, Sexta Sección",Mendoza,Mendoza,Argentina,M5500,NaN,Plus,643.0,3178204.11,Teacher,Single,College Prep,2020-04-14 19:24:07,SUC-04MF924X,Active,2020-12-03 19:24:07,False,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/customers.csv,2026-09-28 19:22:19.503896-03:00,customers
8,CLI-W03121292Q9V,32972346,DNI,Adria

In [35]:
q("SELECT len(last_name) as len, min(last_name), max(last_name), count(*) AS n FROM bronze.customers GROUP BY 1 ORDER BY len")

,len,min(last_name),max(last_name),n
0,9,Cruz Cruz,Vera Vera,1483
1,10,Cruz Gómez,Vera Silva,5253
2,11,Acosta Díaz,Vera Vargas,13400
3,12,Acosta Gómez,Álvarez Vera,22240
4,13,Acosta Acosta,Álvarez Silva,27586
5,14,Acosta Benítez,Álvarez Vargas,28715
6,15,Acosta González,Álvarez Álvarez,23117
7,16,Acosta Domínguez,Álvarez Valencia,15208
8,17,Aguilar Contreras,Álvarez Rodríguez,8682
9,18,Castillo Contreras,Valencia Rodríguez,3177


In [36]:
q("SELECT * FROM bronze.customers WHERE last_name in ('Cruz Cruz', 'Vera Vera', 'Acosta Acosta', 'Álvarez Álvarez', 'Rodríguez Rodríguez', 'Contreras Contreras')")

,customer_id,document_number,document_type,first_name,last_name,date_of_birth,gender,email,mobile_phone,landline_phone,address,city,state,country,postal_code,detected_accent,segment,credit_score,estimated_monthly_income,occupation,marital_status,education_level,registration_date,registration_branch_id,customer_status,last_updated,accepts_marketing,_source_file,_ingested_at,_source_table
0,CLI-9I9TTVRHGQT7,97868953,DNI,Fernando,Contreras Contreras,1993-09-15,M,fernando.contreras@yahoo.com,+54 9 16 6189 9635,NaN,"Circuito Independencia 11, Centro",Guadalajara,Jalisco,México,44100,NaN,Plus,673.0,74899.18,Student,Married,College Prep,2024-12-25 04:39:19,SUC-RYFDRSX3,Active,2025-04-02 04:39:19,False,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/customers.csv,2026-09-28 19:22:19.503896-03:00,customers
1,CLI-V0RQPRIPW55G,93057410,DNI,Luis,Acosta Acosta,1970-06-28,O,luisacosta@hotmail.com,+54 9 89 9342 6479,NaN,"Camino Avenida Callao 178, Nueva Córdoba",Córdoba,Córdoba,Argentina,X5000,argentine,Basic,653.0,974374.7,Student,Widowed,High School,2021-06-16 15:20:51,SUC-6RJUOLTG,Active,2022-02-23 15:20:51,False,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/customers.csv,2026-09-28 19:22:19.503896-03:00,customers
2,CLI-ZDQJD49YKLCG,00427472,DNI,Patricia Mónica,Contreras Contreras,1945-06-23,F,patricia.contreras960@icloud.com,+54 9 99 7485 4684,NaN,"Andador San Juan 805, Centro",Puebla,Puebla,México,72000,NaN,Basic,585.0,30948.19,Independent Professional,Married,University,2022-08-26 17:32:34,SUC-NQNIHMHM,Closed,2023-05-24 17:32:34,True,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/customers.csv,2026-09-28 19:22:19.503896-03:00,customers
3,CLI-05RHW7DX6HKL,61798188,DNI,Catalina Catalina,Álvarez Álvarez,1996-06-26,F,calvarez@outlook.com,+54 9 69 3283 4807,+54 78 6509 2614,"Boulevard Calle Alsina 759, San Martín",Mendoza,Mendoza,Argentina,NaN,NaN,Basic,610.0,963671.91,Manager,Single,NaN,2022-02-19 21:14:40,SUC-TAHKL85Z,Active,2022-10-12 21:14:40,False,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/customers.csv,2026-09-28 19:22:19.503896-03:00,customers
4,CLI-ICQJC2Z28YN7,98306507,DNI,Catalina,Contreras Contreras,1953-10-03,F,catalina.contreras@outlook.com,+54 9 58 3388 9593,NaN,"Camino Del Valle 902, Centro",Tijuana,Baja California,México,22000,NaN,Basic,642.0,18290.84,Doctor,Single,Elementary,2020-12-16 11:17:39,SUC-9QS3J4VH,Active,2021-10-17 11:17:39,True,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/customers.csv,2026-09-28 19:22:19.503896-03:00,customers
5,CLI-6PIX0V9T17AE,14630552,DNI,Victoria,Rodríguez Rodríguez,1945-11-19,O,victoriarodriguez@icloud.com,+54 9 58 9895 1567,+54 82 6670 8221,"Ruta Calle Florida 58, Jardín",Córdoba,Córdoba,Argentina,NaN,NaN,Basic,684.0,NaN,Employee,Single,University,2020-03-06 13:14:29,SUC-YE4YM9WN,Active,2020-07-23 13:14:29,False,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/customers.csv,2026-09-28 19:22:19.503896-03:00,customers
6,CLI-B49S5RGI4WD5,62999237,DNI,Marta,Vera Vera,1978-04-10,F,mvera@outlook.com,+54 9 22 2271 3650,+54 94 2923 8334,"Boulevard Avenida Belgrano 91, Quinta Sección",Mendoza,Mendoza,Argentina,M5500,argentine,Basic,603.0,286318.8,Consultant,Divorced,College Prep,2020-01-02 09:43:44,SUC-56UIY81S,Active,2020-05-21 09:43:44,False,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/customers.csv,2026-09-28 19:22:19.503896-03:00,customers
7,CLI-YD1MA84TMYXN,44275188,DNI,Carolina,Contreras Contreras,1971-12-28,F,carolinacontreras@gmail.com,+54 9 66 8660 6425,+54 73 9888 3569,"Paseo Las Flores 765, Centro",Guadalajara,Jalisco,México,44100,mexican,Plus,635.0,58000.12,Director,Divorced,College Prep,2022-08-14 04:24:22,SUC-4PHH9AHK,Active,2023-01-02 04:24:22,True,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/customers.csv,2026-09-28 19:22:19.503896-03:00,customers
8,CLI-K0Y4NATY2SB9,7388672,CE,Raquel Mercedes,Rodríguez Rodríguez,1961-11-27,F,raquel.rodriguez@live.com,+57 31

In [37]:
# Users note: i want to do this type of side by side comparisson of bronze and silver
print("Bronze raw values:")
display(q("SELECT typeof(customer_status) , len(customer_status) as len, customer_status, count(*) AS n FROM bronze.customers GROUP BY all ORDER BY customer_status"))

print("Silver parsed values:")
display(q("SELECT typeof(customer_status) , len(customer_status) as len, customer_status, count(*) AS n FROM silver.dim_customers GROUP BY all ORDER BY customer_status"))

Bronze raw values:


,typeof(customer_status),len,customer_status,n
0,VARCHAR,6,Active,127700
1,VARCHAR,6,Closed,2979
2,VARCHAR,8,Inactive,14914
3,VARCHAR,9,Suspended,4407


Silver parsed values:


,typeof(customer_status),len,customer_status,n
0,VARCHAR,6,Active,127700
1,VARCHAR,6,Closed,2979
2,VARCHAR,8,Inactive,14914
3,VARCHAR,9,Suspended,4407


In [39]:
con.close()
print("OK")

OK
